# Filtering Columns and Rows:

## Columns to retain in performance data

### Identifiers and filters
- new_la_code/la_name - for joining to deprivation dataset
- school_urn/school_name - unique school id
- establishment_type_group - academy, independent, maintained
- sex
- disadvantage_status
- first_language
- breakdown_topic


### Outcome measures
- attainment8_average
- progress8_average
- progress8_lower_95_ci / progress8_upper_95_ci
- ebacc_aps_average

### Subject-specific (for Q3)
- attainment8eng_average, attainment8mat_average, attainment8ebacc_average
- progress8eng_average, progress8mat_average, progress8ebacc_average, progress8open_average
- ebacceng_95_percent, ebaccmat_95_percent, ebaccsci_95_percent, ebacchum_95_percent, ebacclan_95_percent

### Pupil characteristics
- pupil_count


## Columns to retain in deprivation dataset

### Identifier
- Local Authority District code (2019)

### Deprivation (for Q1)
- Index of Multiple Deprivation (IMD) Score

### Specific Deprivation Factors (for Q2)
- Income Score (rate)
- Employment Score (rate)
- Health Deprivation and Disability Score
- Crime Score
- Barriers to Housing and Services Score
- Living Environment Score
- Income Deprivation Affecting Children Index (IDACI) Score (rate)
- Children and Young People Sub-domain Score


In [19]:
import numpy as np
import pandas as pd

performance_data = pd.read_csv('/Users/aislinglangston/Desktop/CFG_group_project/202425_performance_tables_schools_final.csv')

iod_data = pd.read_csv('/Users/aislinglangston/Desktop/CFG_group_project/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv')

base_cols = [
    'school_urn', 'school_name', 'la_name', 'new_la_code',
    'establishment_type_group', 'pupil_count',
    'attainment8_average', 'progress8_average', 'ebacc_aps_average'
]

subject_cols = [
    'attainment8eng_sum', 'attainment8mat_sum',
    'attainment8ebacc_sum', 'attainment8open_sum',
    'ebacceng_94_percent', 'ebaccmat_94_percent',
    'ebaccsci_94_percent', 'ebacchum_94_percent',
    'ebacclan_94_percent'
]

# Base filter conditions
total_filter = (
    (performance_data['breakdown_topic'] == 'Total') &
    (performance_data['sex'] == 'Total') &
    (performance_data['disadvantage_status'] == 'Total') &
    (performance_data['first_language'] == 'Total')
)

gender_filter = (
    (performance_data['breakdown_topic'] == 'Sex') &
    (performance_data['disadvantage_status'] == 'Total') &
    (performance_data['first_language'] == 'Total')
)

eal_filter = (
    performance_data['breakdown_topic'] == 'First language'
)

performance_total  = performance_data[total_filter][base_cols + subject_cols].copy()
perf_gender = performance_data[gender_filter][base_cols + ['sex']].copy()
perf_eal    = performance_data[eal_filter][base_cols + ['first_language']].copy()

print("---NEW PERFORMANCE DATA STRUCTURE ------")
print(performance_total.head(10))
print(performance_total.info())
print(performance_total.columns)

iod_la = iod_data.groupby(
    ['Local Authority District code (2019)', 'Local Authority District name (2019)']
).agg(
    imd_score          = ('Index of Multiple Deprivation (IMD) Score', 'mean'),
    income_score       = ('Income Score (rate)', 'mean'),
    employment_score   = ('Employment Score (rate)', 'mean'),
    health_score       = ('Health Deprivation and Disability Score', 'mean'),
    crime_score        = ('Crime Score', 'mean'),
    barriers_score     = ('Barriers to Housing and Services Score', 'mean'),
    living_env_score   = ('Living Environment Score', 'mean'),
    idaci_score        = ('Income Deprivation Affecting Children Index (IDACI) Score (rate)', 'mean'),
    children_subdomain = ('Children and Young People Sub-domain Score', 'mean'),
).reset_index()

iod_la.rename(columns={
    'Local Authority District code (2019)': 'la_code'
}, inplace=True)

print("---NEW DEPRIVATION DATA STRUCTURE ------")
print(iod_la.head(10))
print(iod_la.info())

---NEW PERFORMANCE DATA STRUCTURE ------
    school_urn                      school_name         la_name new_la_code  \
0       100544               David Game College  City of London   E09000001   
7       100001  City of London School for Girls  City of London   E09000001   
14      100003            City of London School  City of London   E09000001   
21      137181                  The UCL Academy          Camden   E09000007   
28      100049                Haverstock School          Camden   E09000007   
35      100050           Parliament Hill School          Camden   E09000007   
42      100051               Regent High School          Camden   E09000007   
49      100052                 Hampstead School          Camden   E09000007   
56      100053           Acland Burghley School          Camden   E09000007   
63      100054      The Camden School for Girls          Camden   E09000007   

   establishment_type_group pupil_count attainment8_average progress8_average  \
0       

# Missing Data

On initial visual review of the performance dataset, there is a large amount of missing data, with the following notation:

- `z` denoting missing/suppressed data
- `c` denoting counts too low to report

While this implies a certain amount of missingness analysis pre-completed, it is important to evaluate which fields and records have missing data, how much, and of what type, in order to determine the appropriate way of dealing with this missing data.

In [21]:
# MISSING DATA - PERFORMANCE
print("--- MISSING DATA INITIAL ANALYSIS---")
# Per column counts
c_by_col = (performance_total == 'c').sum()
z_by_col = (performance_total == 'z').sum()

# Show only columns that actually have them
performance_total_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
performance_total_suppression_summary = performance_total_suppression_summary[
    (performance_total_suppression_summary['c_count'] > 0) | 
    (performance_total_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance total supression summary: \n {performance_total_suppression_summary}")


# Gender 
# Per column counts
c_by_col = (perf_gender == 'c').sum()
z_by_col = (perf_gender == 'z').sum()

# Show only columns that actually have them
perf_gender_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
perf_gender_suppression_summary = perf_gender_suppression_summary[
    (perf_gender_suppression_summary['c_count'] > 0) | 
    (perf_gender_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance gender supression summary: \n {perf_gender_suppression_summary}")

# EAL
# Per column counts
c_by_col = (perf_eal == 'c').sum()
z_by_col = (perf_eal == 'z').sum()

# Show only columns that actually have them
perf_eal_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
perf_eal_suppression_summary = perf_eal_suppression_summary[
    (perf_eal_suppression_summary['c_count'] > 0) | 
    (perf_eal_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance EAL supression summary: \n {perf_eal_suppression_summary}")

# MISSING DATA - IOD
print(f"\nIOD Missing data summary: {iod_data.isnull().sum().sum()} total missing values")

--- MISSING DATA INITIAL ANALYSIS---

Performance total supression summary: 
                       c_count  z_count
ebacchum_94_percent       608     1311
ebaccsci_94_percent       580     1247
ebacclan_94_percent       566     1364
attainment8_average       394      625
ebacc_aps_average         394      625
attainment8eng_sum        394      625
attainment8mat_sum        394      625
attainment8ebacc_sum      394      625
attainment8open_sum       394      625
ebacceng_94_percent       394      625
ebaccmat_94_percent       394      625
pupil_count                 2       34
progress8_average           0     5755

Performance gender supression summary: 
                      c_count  z_count
attainment8_average     2339     1273
ebacc_aps_average       2339     1273
pupil_count              751      721
progress8_average          0    11510

Performance EAL supression summary: 
                      c_count  z_count
attainment8_average      849     2081
ebacc_aps_average        849 

## Missing Data Analysis

### Performance Dataset Missing Data

As can be seen from the above, there is a significant amount of missing data in the performance dataset.

progress8_average was entirely `z` denoted, with no numerical values at all, which is driven by Covid disruption meaning that the benchmark to assess progress was not available. This means that this field for this year is meaningless as an outcome. 

For `attainment8_average` and `ebacc_aps_average` there were 394 `c` entries (which equals ~6.8% of 5,755 rows), and 625 `z` entries (~10.9% of total rows). This means that there is small cohort supression, affecting small (and often independent) schools, as evidenced by the `c` entries, while the quantity of `z` entries implies that there may be other structural corrlations with `z` entries. These missing values for these fields are thus missing not at random (MNAR).

`perf_gender` shows higher levels of `c` than `total` which is driven by the fact that for mixed schools the cohorts are divided, making them more likely to fall below the threshold below which `c` is reported.

`perf_eal` also shows higher levels of `c`, alhough this is likely driven almost entirely by the EAL group rather than the non-EAL group; schools with very few EAL pupils will be suppressed.


### IOD Dataset Missing Data

This dataset is commplete, so no further action is needed to deal with missing data.


## Missing Data Methdology

Findings generalise most reliably to mainstream state secondary schools with more than 100 pupils. Conclusions should therefore not be extrapolated to small or independent schools.

The most principled approach to dealing with this missing data is therefore to restrict the entire analysis to large, mainstream state-funded schools.

So, the following methodology is appropriate:

1. Convert suppression codes to NaN
2. Document what is going to be dropped
3. Restrict analysis to large, mainstream state-funded schools
4. Evaluate and handle any remaining NaN

### Converting suppression codes to NaN

In [22]:
suppression_codes = ['c', 'z']

# Apply to all three dataframes
for df in [performance_total, perf_gender, perf_eal]:
    df.replace(suppression_codes, np.nan, inplace=True) # replace c and z with NaN

# Now convert numeric columns that were blocked by string codes
numeric_cols = [
    'attainment8_average', 'ebacc_aps_average', 
    'pupil_count', 'progress8_average',
    'attainment8eng_average', 'attainment8mat_average',
    'progress8eng_average', 'progress8mat_average',
    'progress8ebacc_average', 'progress8open_average',
    'ebacceng_95_percent', 'ebaccmat_95_percent',
    'ebaccsci_95_percent', 'ebacchum_95_percent', 
    'ebacclan_95_percent'
]

for df in [performance_total, perf_gender, perf_eal]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce') # convert columns to numeric values. E.g. 1 instead of "1"



/var/folders/hc/5z1l136x7b12q7qzw10d_3kw0000gn/T/ipykernel_5161/2388975954.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(suppression_codes, np.nan, inplace=True) # replace c and z with NaN


### Documenting what is to be dropped

In [23]:
def document_exclusions(df, name):
    total = len(df)
    missing_att8 = df['attainment8_average'].isnull().sum()
    missing_prog8 = df['progress8_average'].isnull().sum()
    
    print(f"\n{name}:")
    print(f"  Total rows: {total}")
    print(f"  Missing attainment8: {missing_att8} ({missing_att8/total*100:.1f}%)")
    print(f"  Missing progress8:   {missing_prog8} ({missing_prog8/total*100:.1f}%)")
    
    # School types being excluded
    print(f"  School types with missing attainment8:")
    print(df[df['attainment8_average'].isnull()]
          ['establishment_type_group'].value_counts().to_string())

document_exclusions(performance_total, "perf_total")
document_exclusions(perf_gender, "perf_gender")
document_exclusions(perf_eal, "perf_eal")


perf_total:
  Total rows: 5755
  Missing attainment8: 1019 (17.7%)
  Missing progress8:   5755 (100.0%)
  School types with missing attainment8:
establishment_type_group
Independent special schools             349
Community special school                193
Converter academies - special school    147
Independent schools                     128
Free schools                             51
Foundation special school                40
Free schools - special school            38
Sponsored academies - special school     28
Non-maintained special schools           26
Sponsored academies                      10
Converter academies                       6
Voluntary controlled school               2
University technical colleges (UTCs)      1

perf_gender:
  Total rows: 11510
  Missing attainment8: 3612 (31.4%)
  Missing progress8:   11510 (100.0%)
  School types with missing attainment8:
establishment_type_group
Independent special schools             1068
Independent schools                   

As can be seen from the output above, the missing data is concentrated in special schools, and independent schools, likely due to small cohorts and, in the case of special schools, potentially little or no uptake of GCSE qualifications.

### Restrict to large, mainstream, state-funded schools

In [24]:
mainstream_schools = [
    'Community school',
    'Voluntary aided school',
    'Voluntary controlled school',
    'Foundation school',
    'Converter academies',
    'Sponsored academies',
    'Free schools',
    'University technical colleges (UTCs)',
    'City technology college',
    'Studio schools'
]

def clean_performance(df):
    perf_df_clean = df.copy()
    
    # Keep only mainstream state schools
    perf_df_clean = perf_df_clean[perf_df_clean['establishment_type_group'].isin(mainstream_schools)]

    # Keep only schools with a minimum viable cohort (20?)
    perf_df_clean = perf_df_clean[pd.to_numeric(perf_df_clean['pupil_count'], errors='coerce') >= 20]

    # Drop rows missing attainment8_average
    perf_df_clean = perf_df_clean.dropna(subset=['attainment8_average'])

    # Print size of cleaned dataset
    print(f"Rows retained: {len(perf_df_clean)} of {len(df)}")
    return perf_df_clean

perf_total_clean   = clean_performance(performance_total)
perf_gender_clean  = clean_performance(perf_gender)
perf_eal_clean     = clean_performance(perf_eal)

Rows retained: 3277 of 5755
Rows retained: 6154 of 11510
Rows retained: 1742 of 5755


Having completed the main, systematic cleaning, we consider the amount of data retained. After restricting to mainstream state-funded schools with a minimum cohort of 20 pupils, 3,277 schools were retained for the primary analysis (56.9% of original dataset). The gender breakdown retained 
6,154 school-sex observations (53.4%), and the EAL breakdown retained 1,742 observations (30.3%).

The restriction of the EAL dataset has implications for the power of our conclusions in research question 5. Findings for this question should  be interpreted with caution, as they are likely to overrepresent schools in more diverse areas where EAL cohorts are large enough to report.

### Evaluate and Handle Remaining NaN

In [26]:
numeric_only = perf_total_clean.select_dtypes(include='number')
remaining_missing = numeric_only.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("No remaining missing values in numeric columns")
else:
    total_rows = len(perf_total_clean)
    remaining_df = pd.DataFrame({
        'missing_count': remaining_missing,
        'missing_percent': (remaining_missing / total_rows * 100).round(2)
    })
    print(remaining_df)

for name, df in [('perf_gender_clean', perf_gender_clean), 
                 ('perf_eal_clean', perf_eal_clean)]:
    print(f"\n=== {name} ===")
    numeric_only = df.select_dtypes(include='number')
    remaining = numeric_only.isnull().sum()
    remaining = remaining[remaining > 0]
    
    if len(remaining) == 0:
        print("No remaining missing values")
    else:
        print(pd.DataFrame({
            'missing_count': remaining,
            'missing_percent': (remaining / len(df) * 100).round(2)
        }))

                   missing_count  missing_percent
progress8_average           3277            100.0

=== perf_gender_clean ===
                   missing_count  missing_percent
progress8_average           6154            100.0

=== perf_eal_clean ===
                   missing_count  missing_percent
progress8_average           1742            100.0


For all three progress datasets, the only NaN remain in the progress8_average field, which should therefore be dropped.